# Clustering with logical constraints

Unsupervised learning: 200 unlabelled points are split into four clusters. The knowledge base says what a good clustering is (every point is in a cluster, every cluster has points, close points go together, far points apart) and training finds a membership predicate that satisfies it.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
import sltn
from sltn import Signature, Interpretation, KB, kb_describe, kb_backward
from sltn.interpretation.operators import AggregPMean, AggregPMeanError

torch.manual_seed(0)
np.random.seed(0)
device = "cpu"

## Data

Four Gaussian clouds of 50 points, one per quadrant of $[-1, 1]^2$. The cloud of each point is used only for the plots.

In [ ]:
n_clusters, n_per_cluster, margin = 4, 50, 0.2
means = [np.random.uniform([lo_x + margin, lo_y + margin], [lo_x + 1 - margin, lo_y + 1 - margin])
         for lo_x, lo_y in [(-1, -1), (0, -1), (-1, 0), (0, 0)]]
clouds = [np.random.multivariate_normal(m, 0.01 * np.eye(2), n_per_cluster) for m in means]
points = torch.tensor(np.concatenate(clouds), dtype=torch.float32, device=device)

plt.figure(figsize=(4, 4))
for cloud in clouds:
    plt.scatter(cloud[:, 0], cloud[:, 1], s=10)
plt.title("Data (colours not used in training)")
plt.show()

## 1 — Signature

- sorts `Point` and `Cluster`;
- a predicate `C(x, c)`: point `x` belongs to cluster `c`, the network;
- predicates `Close(x, y)` and `Far(x, y)` on pairs of points, fixed from their distance;
- variables `x`, `y` over points and `c` over clusters.

In [ ]:
sig = Signature("clustering")
sig.sort("Point")
sig.sort("Cluster")
sig.predicate("C", ["Point", "Cluster"])
sig.predicate("Close", ["Point", "Point"])
sig.predicate("Far", ["Point", "Point"])
sig.variable("x", "Point")
sig.variable("y", "Point")
sig.variable("c", "Cluster")
print(sig)

## 2 — Knowledge base

```
∀ x: ∃ c: C(x, c)                                  every point is in some cluster
∀ c: ∃ x: C(x, c)                                  every cluster has some point
∀ c, x, y | Close(x, y): (C(x, c) <-> C(y, c))     close points are in the same clusters
∀ c, x, y | Far(x, y): ¬(C(x, c) & C(y, c))        far points are not in the same cluster
```

The first two rule out trivial solutions (no point anywhere, or everything in one cluster). The last two use **guards** (Tutorial 2 §4): the ∀ ranges only over the pairs that are close, or far. Here `x`, `y` are both all 200 points, so `∀ x, y` ranges over all 40 000 pairs and the guard picks the ones that count.

In [ ]:
kb = KB([
    sig.parse("∀ x: ∃ c: C(x, c)"),
    sig.parse("∀ c: ∃ x: C(x, c)"),
    sig.parse("∀ c, x, y | Close(x, y): (C(x, c) <-> C(y, c))"),
    sig.parse("∀ c, x, y | Far(x, y): ¬(C(x, c) & C(y, c))"),
])
kb_dict = kb_describe(kb)

## 3 — Model

An MLP from a point to a softmax over the four clusters; `C(x, c)` is the probability of cluster `c`, with `c` a one-hot vector. The softmax makes each point belong to one cluster.

In [ ]:
class ClusterMLP(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(2, 16), torch.nn.ELU(),
            torch.nn.Linear(16, 16), torch.nn.ELU(),
            torch.nn.Linear(16, 16), torch.nn.ELU(),
            torch.nn.Linear(16, 4), torch.nn.Softmax(dim=-1),
        )

    def forward(self, x, c):
        return (self.net(x) * c).sum(-1, keepdim=True)

## 4 — Interpretation

`Close` is true below distance 0.2, `Far` above 0.5, through a steep sigmoid. The thresholds are fixed: if they were learned, the guarded clauses could be satisfied by moving them until no pair is close or far.

The operators (Tutorial 3):
- ∀ is pMeanError with p = 4, stricter than the default;
- ∃ is pMean, with p changed during training (below);
- the clauses are combined with pMean (p = 2).

In [ ]:
interp = Interpretation(sig).to(device)
interp.logic["∀"] = AggregPMeanError(p=4)
interp.logic["∃"] = AggregPMean(p=1)
interp.logic["SatAgg"] = AggregPMean(p=2)

interp["Point"] = sltn.Type("Point", shape=(2,))
interp["Cluster"] = sltn.Type("Cluster", shape=(4,))
interp["C"] = ClusterMLP()
distance = lambda a, b: torch.norm(a - b, dim=-1, keepdim=True)
interp["Close"] = lambda a, b: torch.sigmoid(50 * (0.2 - distance(a, b)))
interp["Far"] = lambda a, b: torch.sigmoid(50 * (distance(a, b) - 0.5))

interp["x"] = points
interp["y"] = points
interp["c"] = torch.eye(4, device=device)
print(interp)

## 5 — Training

All the points at every step, no minibatches. `kb_backward` (Tutorial 4) computes the loss $1 - \mathrm{SatAgg}$ and also returns the truth value of each clause.

The ∃ starts lenient, p = 1 (the mean), so that every cluster receives a gradient; with p = 6 it approaches the maximum: "some point is in the cluster" asks one point to be in it with high truth.

In [ ]:
optimizer = torch.optim.Adam(interp.parameters(), lr=0.01)

for epoch in range(1000):
    interp.logic["∃"].p = 1 if epoch <= 100 else 6
    optimizer.zero_grad()
    stats = kb_backward(optimizer, interp, kb_dict, aggregator="satagg")
    optimizer.step()
    if epoch % 200 == 0 or epoch == 999:
        print(f"epoch {epoch:4d} | " + " ".join(f"{name}={sat:.3f}" for name, sat in stats["sats"].items()))

## 6 — Results

Each point coloured by its most likely cluster, and the truth of `C(x, c)` for each cluster.

In [ ]:
interp.eval()
with torch.no_grad():
    membership = interp["C"].net(points).cpu()
xy = points.cpu()

fig, axes = plt.subplots(1, 5, figsize=(17, 3.3))
axes[0].scatter(xy[:, 0], xy[:, 1], c=membership.argmax(-1), cmap="tab10", vmin=0, vmax=9, s=10)
axes[0].set_title("Assigned cluster")
for k in range(4):
    sc = axes[k + 1].scatter(xy[:, 0], xy[:, 1], c=membership[:, k], vmin=0, vmax=1, s=10)
    axes[k + 1].set_title(f"C(x, c{k})")
fig.colorbar(sc, ax=axes[1:])
plt.show()

true_cloud = torch.arange(4).repeat_interleave(n_per_cluster)
print("points per cluster:", membership.argmax(-1).bincount(minlength=4).tolist())
print("clouds split across clusters:", [membership.argmax(-1)[true_cloud == k].unique().tolist() for k in range(4)])

Each cloud ends up in its own cluster. The first two clauses stop at 0.79 even with crisp memberships: the pMean ∃ of (1, 0, 0, 0) with p = 6 is $(1/4)^{1/6} \approx 0.79$, because it is a smooth maximum. The far clause stays below 1 for a similar reason, the soft memberships at the cloud borders.

## Summary

- The knowledge base describes a good clustering, and training finds a predicate that satisfies it.
- Guards restrict a ∀ to the pairs that are close, or far; they are fixed here, since learned guards could empty themselves.
- The quantifier aggregators are part of the model: ∃ with a growing p goes from "on average" to "at least one".
- `kb_backward` gives the truth of each clause, to see which ones are hard to satisfy.